# Belajar alur AES: teks → empat skor

Jalankan cell **berurutan dari atas**, tanpa menambahkan cell dari percakapan lama. Pilih **CPU**, **Runtime version 2026.07 (Python 3.12)** sebelum mulai. Default notebook memeriksa data; skor guru yang kosong tetap kosong.

Proyek memakai empat aspek (*trait*): **isi, organisasi, kebahasaan, mekanik**, kategori 1–5. Tahapnya: ambil kode → siapkan Python AES → pilih CSV/JSON → audit → baca satu esai dan rubrik → cleaning → jalur model opsional → kesiapan training.

Pustaka AES berjalan dalam **lingkungan Python terpisah**, yaitu folder paket milik proyek. Setiap langkah perhitungan memakai proses baru sehingga NumPy yang sudah termuat di memori Colab tidak ikut terbawa. Setup CPU pertama mengunduh paket; berikutnya memakai lingkungan yang sama. Instalasi tidak mengubah paket Python bawaan Colab.

CSV/JSON siswa tidak ada di repo publik. File dan semua hasil inspeksi tetap lokal. Jangan simpan notebook yang sudah berisi teks siswa ke GitHub. Pengecekan model asli dan unit test tersedia lewat pilihan pada cell 1; keduanya berbeda dari training dengan skor guru.

## 1. Lokasi proyek dan pilihan belajar

Cell ini mengambil branch `codex/aes-learning` bila proyek belum tersedia. Jika checkout Colab sudah ada, kode diperbarui dengan *fast-forward*: menambahkan versi terbaru tanpa menulis ulang riwayat. Perubahan file tracked milik pengguna tidak ditimpa.

Biarkan `MODE="inspect"`, `RUN_ENCODERS=False`, dan `RUN_TESTS=False` untuk putaran pertama. Setelah audit dipahami, ubah `RUN_ENCODERS=True` untuk mengunduh BGE-M3/RoBERTa asli dan melihat token, vektor, serta output UNTRAINED. CPU dapat menjalankannya, tetapi lebih lambat; setup notebook ini memasang PyTorch CPU meskipun runtime memiliki GPU.

In [ ]:
from pathlib import Path
import os, sys, json, subprocess, hashlib

MODE = "inspect" #@param ["inspect", "learning", "proposal"]
RUN_ENCODERS = False #@param {type:"boolean"}
RUN_TESTS = False #@param {type:"boolean"}
RUN_AC2_REFERENCE = False #@param {type:"boolean"}

if sys.version_info[:2] != (3, 12):
    raise RuntimeError("Pilih Runtime version 2026.07, Python 3.12, CPU sebelum setup.")
IN_COLAB = Path('/content').is_dir()
if os.environ.get('AES_NOTEBOOK_ROOT'):
    ROOT = Path(os.environ['AES_NOTEBOOK_ROOT']).resolve()
elif IN_COLAB:
    ROOT = Path('/content/Automated_Essay_Scoring')
else:
    ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                 if (p/'configs/default.json').is_file()), None)
    if ROOT is None:
        raise FileNotFoundError('Buka notebook dari checkout proyek atau atur AES_NOTEBOOK_ROOT.')

if IN_COLAB and not ROOT.exists():
    subprocess.run(['git', 'clone', '--branch', 'codex/aes-learning', '--single-branch',
                    'https://github.com/MalikHariyanto/Automated_Essay_Scoring.git', str(ROOT)], check=True)
elif IN_COLAB:
    dirty = subprocess.check_output(['git', '-C', str(ROOT), 'status', '--porcelain',
                                     '--untracked-files=no'], text=True).strip()
    if dirty:
        raise RuntimeError('Checkout memiliki perubahan tracked. Simpan perubahan tersebut sebelum memperbarui kode.')
    branch = subprocess.check_output(['git', '-C', str(ROOT), 'branch', '--show-current'], text=True).strip()
    if branch != 'codex/aes-learning':
        raise RuntimeError('Checkout memakai branch lain; gunakan branch codex/aes-learning.')
    subprocess.run(['git', '-C', str(ROOT), 'fetch', 'origin', 'codex/aes-learning'], check=True)
    subprocess.run(['git', '-C', str(ROOT), 'merge', '--ff-only', 'FETCH_HEAD'], check=True)

if not (ROOT/'scripts/setup_notebook.py').is_file():
    raise FileNotFoundError('Checkout belum memuat setup notebook terbaru.')
os.chdir(ROOT)
OUTPUT = ROOT/'outputs'
OUTPUT.mkdir(exist_ok=True)
ENV_DIR = OUTPUT/'notebook_env'
AES_PYTHON = ENV_DIR/'bin/python'
WORKER_ENV = os.environ.copy()
WORKER_ENV.pop('PYTHONPATH', None)
WORKER_ENV.pop('PYTHONHOME', None)
WORKER_ENV.update(HF_HOME=os.environ.get('HF_HOME', str(OUTPUT/'cache/huggingface')), OMP_NUM_THREADS='2', MKL_NUM_THREADS='2')
print('Python notebook:', sys.version.split()[0])
print('Folder proyek:', ROOT)
print('Mode:', MODE, '| Model asli:', RUN_ENCODERS, '| Unit test:', RUN_TESTS)

## 2. Setup dan pengecekan pustaka

**Pustaka** ialah komponen yang dipakai program. Lingkungan dibuat tanpa `ensurepip`; jika pip target belum ada, pip bawaan Colab memasangnya melalui opsi resmi `--python`. Lingkungan parsial akibat percobaan sebelumnya dipertahankan dan dipulihkan. Setup memasang versi dari `requirements.lock.txt` pada `outputs/notebook_env`, kemudian benar-benar mengimpor NumPy, SciPy, scikit-learn, PyTorch, Transformers, dan FlagEmbedding di proses Python baru.

Log pemasangan disimpan di `outputs/notebook_setup.log`. Jika gagal, error tetap menghentikan cell dan bagian akhir log ditampilkan. Setup tidak dianggap berhasil hanya karena nama paket proyek terpasang. Saat berhasil, lanjut ke cell berikut; tidak perlu cell reinstall atau restart tambahan.

In [ ]:
setup_log = OUTPUT/'notebook_setup.log'
with setup_log.open('w', encoding='utf-8') as log:
    setup_result = subprocess.run([sys.executable, str(ROOT/'scripts/setup_notebook.py'),
                                   '--root', str(ROOT), '--environment', str(ENV_DIR)],
                                  cwd=ROOT, env=WORKER_ENV, stdout=log, stderr=subprocess.STDOUT)
if setup_result.returncode:
    print('\n'.join(setup_log.read_text().splitlines()[-35:]))
    raise subprocess.CalledProcessError(setup_result.returncode, setup_result.args)
setup_proof = json.loads((ENV_DIR/'aes_setup.json').read_text())
print('Python AES:', setup_proof['python'])
for paket, versi in setup_proof['versions'].items():
    print(paket + ':', versi)
print('IMPORT OK:', setup_proof['imports_ok'])
print('Checkpoint sudah dimuat:', setup_proof['checkpoint_loaded'])
print('Log setup:', setup_log)

def run_aes(arguments):
    return subprocess.run([str(AES_PYTHON), *map(str, arguments)],
                          cwd=ROOT, env=WORKER_ENV, check=True)

## 3. Masukkan CSV dan JSON asli

Jika kedua file belum tersedia, Colab membuka pemilih file pada cell ini. Pilih **dataset_aes_multitrait_DRAF.csv** dan **rubrik_multitrait_DRAF.json** dari komputer. Tidak perlu menambahkan cell upload. Jika file sudah ada, cell memakai file tersebut.

Salinan asli disimpan di `data/source/`; label, deskriptor, dan status tidak diubah. Path dapat diganti di dua variabel berikut. Hash adalah sidik jari file untuk memeriksa bahwa sumber yang dipakai tetap sama.

In [ ]:
CSV_PATH = ROOT/'data/source/dataset_aes_multitrait_DRAF.csv'
RUBRIC_PATH = ROOT/'data/source/rubrik_multitrait_DRAF.json'
missing_inputs = [p for p in [CSV_PATH, RUBRIC_PATH] if not p.is_file()]
if missing_inputs and IN_COLAB:
    from google.colab import files
    print('Pilih file:', ', '.join(p.name for p in missing_inputs))
    CSV_PATH.parent.mkdir(parents=True, exist_ok=True)
    os.chdir(CSV_PATH.parent)
    try:
        uploaded = files.upload()
    finally:
        os.chdir(ROOT)
    for target in missing_inputs:
        if target.name in uploaded:
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(uploaded[target.name])
missing_inputs = [p for p in [CSV_PATH, RUBRIC_PATH] if not p.is_file()]
if missing_inputs:
    raise FileNotFoundError('Input belum tersedia: ' + ', '.join(str(p) for p in missing_inputs))
for p in [CSV_PATH, RUBRIC_PATH]:
    print(p.name, '| SHA256:', hashlib.sha256(p.read_bytes()).hexdigest())

cfg = json.loads((ROOT/'configs/default.json').read_text())
cfg.update(csv=str(CSV_PATH.resolve()), rubric=str(RUBRIC_PATH.resolve()),
           output=str(OUTPUT.resolve()), mode=MODE)
cfg['split']['review_file'] = str((ROOT/cfg['split']['review_file']).resolve())
for key, value in cfg.get('sources', {}).items():
    cfg['sources'][key] = str((ROOT/value).resolve()) if value else value
CONFIG_PATH = OUTPUT/'notebook_config.json'
CONFIG_PATH.write_text(json.dumps(cfg, ensure_ascii=False, indent=2), encoding='utf-8')
print('Input siap untuk inspeksi; kelayakan training diperiksa pada audit berikut.')

## 4. Audit aktual dan kesiapan data

**Audit** memeriksa isi file yang sedang dibaca, bukan mengandalkan snapshot lama. Kita mengecek jumlah esai, empat label pada esai yang sama, versi rubrik, status, dan kandidat kemiripan. Ringkasan kelompok usulan tidak membuktikan identitas siswa.

Jika empat skor kosong, program belum mempunyai jawaban guru untuk belajar. Misalnya, output model berupa empat angka belum bisa dibandingkan dengan penilaian guru. Inspeksi tetap berjalan; hanya training/evaluasi supervised yang harus menunggu.

In [ ]:
run_aes([ROOT/'scripts/notebook_step.py', 'audit', '--config', CONFIG_PATH])
report = json.loads((OUTPUT/'audit.json').read_text())
preview = json.loads((OUTPUT/'notebook_preview.json').read_text())
print('Baris:', report['rows'], '| ID esai unik:', report['unique_essay_ids'])
print('Esai dengan empat label terisi:', report['complete_label_rows'])
print('Rubrik disahkan:', report['rubric_confirmed'])
for trait, info in report['traits'].items():
    print(trait, '| missing:', info['missing'], '| invalid:', info['invalid_or_nonfinite'],
          '| noninteger:', info['noninteger'], '| di luar rentang:', info['out_of_range'])
print('Status:', json.dumps(report['statuses'], ensure_ascii=False, indent=2))
print('Exact duplicate:', report['exact_duplicates'])
print('Kandidat near-duplicate untuk ditinjau:', report['near_duplicates'])
print('Kelompok usulan:', report['proposed_group_count'])
print('Cukup lima kelompok untuk lima fold:', report['five_group_folds_possible_by_count'])
print('Kebutuhan sebelum training:')
for reason in report['training_blockers']:
    print('-', reason)
print('EDA teks:', json.dumps(report['eda'], ensure_ascii=False, indent=2))
print('Laporan lengkap:', OUTPUT/'audit.json')

## 5. Satu esai dan deskriptor rubrik

Deskriptor dibaca langsung dari JSON asli. Nilai huruf dan provenance bukan fitur model. Aturan *best fit* dan memilih deskriptor lebih rendah ketika sama kuat merupakan aturan penilaian guru, bukan algoritme pelabelan otomatis.

In [ ]:
essay = preview['essay']
rubric = preview['rubric']
print('essay_id:', essay['essay_id'])
for key in ['scoring_status', 'transcription_status', 'text_completeness',
            'identity_status', 'group_status', 'ready_for_training']:
    print(key + ':', essay.get(key, 'kolom tidak tersedia'))
print('\nTEKS ASLI:\n' + essay['text_raw'])
for trait in rubric['traits']:
    print('\nTRAIT:', trait['name'], '| Kategori:', trait['categories'])
    print('Definisi:', trait['definition'])
    print('Indikator:', trait['indicators'])
    print('Dikecualikan:', trait['exclude'])
    for level, descriptor in trait['levels'].items():
        print(level, descriptor)
print('Aturan guru:', rubric['scoring_rule'])
print('Missing:', rubric['missing_rule'])

## 6. Cleaning sebelum dan sesudah

**Cleaning** merapikan artefak teknis teks. Unicode NFC menyamakan representasi karakter; newline diseragamkan dan karakter kendali teknis dibuang. Kapital, paragraf, salah ejaan, imbuhan, kata penghubung, dan tanda baca tetap dipertahankan agar informasi kebahasaan/mekanik tidak hilang.

Normalisasi kutip/dash/elipsis default mati. File sumber tidak ditimpa; hasil turunan berada di `outputs/cleaning_review.csv`.

In [ ]:
from difflib import unified_diff
text_model = preview['text_model']
print('Sebelum:', repr(essay['text_raw'][:300]))
print('Sesudah:', repr(text_model[:300]))
print('Berubah:', text_model != essay['text_raw'])
print(''.join(unified_diff(essay['text_raw'].splitlines(True), text_model.splitlines(True),
                           fromfile='raw', tofile='model')))
print('Log cleaning:', OUTPUT/'cleaning_review.csv')

## 7. Tokenisasi, window, dan embedding — opsional

**Token IDs** adalah nomor potongan teks dalam kamus tokenizer. **Embedding** adalah vektor angka yang mewakili teks. BGE-M3 memakai maksimum 1024 token termasuk special tokens dan melaporkan bagian terpotong. RoBERTa memakai semua teks melalui **window**, potongan maksimum 512 token termasuk special tokens, dengan overlap 128 token isi.

Untuk checkpoint RoBERTa ini tokenizer slow menghitung dua special tokens; kapasitas isi 510 dan langkah antarwindow 382. Mapping chunk ke esai menjaga agar output tetap satu baris per esai.

Default cell ini dilewati dengan pesan jelas. Untuk menjalankannya, ubah `RUN_ENCODERS=True` di cell 1 dan jalankan kembali dari atas. Unduhan checkpoint sekitar 2,8 GB. Gangguan jaringan dilaporkan; error implementasi tetap muncul.

In [ ]:
model_result = None
if RUN_ENCODERS:
    run_aes([ROOT/'scripts/notebook_step.py', 'models', '--config', CONFIG_PATH])
    model_status = json.loads((OUTPUT/'notebook_models_status.json').read_text())
    if model_status['status'] == 'forward_completed':
        model_result = json.loads((OUTPUT/'forward_inspection.json').read_text())
    else:
        print(model_status)
if model_result is not None:
    print('Revision/config:', model_result['resolved_models'])
    print('BGE tokenisasi:', model_result['bge_tokenization'])
    print('RoBERTa window:', model_result['roberta_windows'])
    print('Esai terpotong:', model_result['truncated_essays'],
          '| Token dipotong:', model_result['removed_tokens'])
    print('Contoh token IDs:', model_result['token_ids_by_encoder'])
    print('Contoh lima elemen vektor:', model_result['vector_first_five'])
else:
    print('Shape/vektor aktual belum tersedia; lihat pilihan RUN_ENCODERS atau status akses checkpoint di atas.')

## 8. Bentuk data dan forward UNTRAINED

**Shape** adalah ukuran susunan angka. Untuk satu esai: BGE `[1,1024]`, RoBERTa `[1,768]`, proyeksi BGE `[1,256]`, gabungan `[1,1024]`, dan output `[1,4]`. Empat kolom output selalu berurutan isi, organisasi, kebahasaan, mekanik.

**Forward** ialah perhitungan teks sampai output. Head baru **UNTRAINED**: belum belajar dari skor guru. Angkanya hanya diagnostik, bisa di luar 1–5, dan tidak boleh dipakai menilai siswa. Shape aktual hanya ditampilkan setelah checkpoint benar-benar dijalankan.

In [ ]:
if model_result is not None:
    print('SHAPES AKTUAL:', model_result['shapes'])
    print('UNTRAINED — keluaran diagnostik:', model_result['untrained_output'])
    print('Frozen/trainable aktual:', model_result['gradient_policy'])
else:
    print('Shape aktual belum dihitung karena jalur model belum dipilih.')

## 9. Frozen, trainable, dan pengujian teknis — opsional

**Frozen** berarti parameter BGE tidak diperbarui. **Trainable** berarti parameter RoBERTa, proyeksi S3, dan head dapat diperbarui saat training. **Gradien** menunjukkan perubahan parameter yang diperlukan untuk mengurangi kesalahan. Empat trait berbagi encoder dan loss; kesalahannya tidak otomatis independen.

**Unit test** memeriksa perilaku bagian kode: urutan target, window, bentuk `[B,4]`, loss yang menolak broadcasting, gradien model tiruan, reload, dan metrik. Fixture ialah input kecil buatan khusus untuk test, bukan dataset penelitian. Lulus unit test bukan bukti akurasi AES.

Aktifkan `RUN_TESTS=True` untuk menjalankan suite. Untuk referensi AC2 terhadap R/irrCAC, aktifkan `RUN_AC2_REFERENCE=True` juga; Colab akan memasang backend R lewat script yang tersedia. Kegagalan dependency R tetap dilaporkan, bukan dianggap lulus. Default keduanya mati agar putaran pertama fokus pada data.

In [ ]:
if RUN_AC2_REFERENCE:
    if IN_COLAB:
        subprocess.run(['bash', str(ROOT/'scripts/setup_colab.sh')],
                       cwd=ROOT, env=WORKER_ENV, check=True)
    else:
        run_aes([ROOT/'scripts/verify_ac2.py', '--require-r'])
else:
    print('Referensi AC2/R belum dijalankan dalam sesi ini.')
if RUN_TESTS:
    run_aes(['-m', 'pytest', '-q', '-rA'])
    print('Unit test selesai; baca passed/failed/skipped pada hasil runner.')
else:
    print('Unit test belum dipilih pada putaran ini.')

## 10. Langkah supervised setelah skor guru tersedia

**Training supervised** memperbarui parameter dari pasangan teks dan skor guru. Pemeriksaan berikut selalu berjalan. Bila data belum layak, hanya tahap supervised yang dilewati dengan daftar kebutuhan konkret; inspeksi tetap selesai.

Syaratnya mencakup empat skor sah, transkripsi lengkap/verified, rubrik disahkan guru, identitas dan hubungan duplikasi diperiksa, serta review terhadap hash CSV terkini. Jangan mengubah flag agar training lolos.

`MODE="learning"` menjalankan latihan singkat yang dipisahkan dari eksperimen proposal. `MODE="proposal"` menyediakan S1/S2/S3, tuning, repeated CV, refit, dan final test. Pilih mode tersebut setelah data dan sumber daya benar-benar siap. Default inspect tidak menghitung loss atau metrik penelitian dari label kosong.

In [ ]:
run_aes([ROOT/'scripts/notebook_step.py', 'supervised', '--config', CONFIG_PATH, '--mode', MODE])
supervised_path = OUTPUT/'notebook_supervised.json'
if supervised_path.is_file():
    print(json.dumps(json.loads(supervised_path.read_text()), ensure_ascii=False, indent=2))
print('Alur notebook selesai. Hasil lokal:', OUTPUT)

## Membaca hasil dan melanjutkan belajar

RMSE dan Pearson memakai prediksi kontinu asli. QWK dan Gwet AC2 memakai kategori setelah clipping dan pemetaan ke kategori terdekat; titik tengah memilih kategori lebih rendah. Metrik dihitung per trait, tanpa output holistik/total. Tanpa label guru, notebook tidak menghasilkan angka evaluasi penelitian.

S1 memakai RoBERTa fine-tuned; S2 BGE frozen + MLP; S3 gabungan RoBERTa dan proyeksi BGE. Diagram dan penjelasan ada di [alur model](https://github.com/MalikHariyanto/Automated_Essay_Scoring/blob/codex/aes-learning/docs/alur_model.md). Keputusan model/protokol dan perbedaan proposal ada di [kontrak implementasi](https://github.com/MalikHariyanto/Automated_Essay_Scoring/blob/codex/aes-learning/docs/implementation_contract.md), [pencocokan sumber](https://github.com/MalikHariyanto/Automated_Essay_Scoring/blob/codex/aes-learning/docs/source_reconciliation.md), serta [review paper](https://github.com/MalikHariyanto/Automated_Essay_Scoring/blob/codex/aes-learning/docs/paper_review.md).

Sesudah checkpoint sungguh dilatih, CLI `python -m aes.cli predict --checkpoint ... --file esai.txt` menghasilkan empat skor kontinu dan empat kategori. Input inference tidak memerlukan skor guru. Jangan menyimpan notebook berisi teks/output siswa ke repository publik.